# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset package using the `mlcroissant` library. We walk through loading dataset metadata, reviewing record sets, extracting tabular data, and performing exploratory data analysis and visualization, all by referencing entities using their Croissant schema `@id` fields.

### Dataset Source
The dataset is defined by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  # Ignore SettingWithCopy warnings

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n\nVersion: {metadata.version}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, their fields, and all referenced `@id` fields for precise data access. 

We enumerate all record sets, and for each record set, list the available fields and their `@id`. This is essential for referencing and extracting data later.

In [ ]:
# List available record sets and their fields (all by @id)

print('Available record sets and their fields (by @id):\n')
record_sets = []
for rs in dataset.record_sets:
    print(f"- Record Set: {rs['@id']} (name: {rs.get('name', '(no name)')})")
    record_sets.append(rs['@id'])
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            fid = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
            print(f"    - Field: {fid}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame using their `@id` values for programmatic, schema-driven access. 

This approach extracts records for *all* record sets (replace `record_sets` variable to select a subset if desired), and allows you to inspect available columns for each by their field/column `@id`.

In [ ]:
# Extract data from each record set using its @id
dataframes = {}

for record_set_id in record_sets:
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set {record_set_id}. Columns:")
            print(list(df.columns))
        else:
            print(f"Record set {record_set_id} is empty.")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For demonstration, pick the first non-empty record set for further analysis
non_empty = [(k,v) for k,v in dataframes.items() if not v.empty]
if non_empty:
    example_record_set_id, example_df = non_empty[0]
    print(f"\nProceeding with example record set: {example_record_set_id}\n")
    print(example_df.head())
else:
    raise RuntimeError("No non-empty record sets available in this dataset!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to the data extracted from one of the record sets. These include filtering records, normalizing numeric fields, and grouping by categorical fields, all referencing fields by their `@id` as specified in the schema.

You may customize the chosen field `@id` below based on available numeric and grouping fields found in the previous step.

In [ ]:
# --- Customize these IDs based on the record set's schema ---
# For demonstration, auto-select the first found numeric field and first grouping field.
import numpy as np

df = example_df.copy()

# Identify candidate fields by dtype
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Try to coerce some columns to numeric just in case
    for col in df.columns:
        try:
            converted = pd.to_numeric(df[col], errors='coerce')
            if converted.notnull().sum() > 0:
                numeric_field_id = col
                df[numeric_field_id] = converted
                break
        except Exception:
            continue

# Pick the first non-numeric string column as grouping/categorical field
for col in df.columns:
    if col != numeric_field_id and df[col].dtype == 'object':
        group_field_id = col
        break

print(f"Numeric field selected (by @id): {numeric_field_id}")
print(f"Group field selected (by @id): {group_field_id}")

if numeric_field_id is None:
    raise RuntimeError("No numeric field found in this record set for demonstration.")

# Example filter: keep records where <numeric_field> > its mean
threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (mean):")
print(filtered_df.head())

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Group by categorical/group field (if available)
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and the group-wise means, all fields specified by their `@id`.

You may re-run with different fields as desired.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of the numeric field
plt.figure(figsize=(8,4))
df[numeric_field_id].dropna().hist(bins=20)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Plot group-wise means if grouping field is available
if group_field_id is not None:
    plt.figure(figsize=(10,4))
    grouped_df.plot(x=group_field_id, y=numeric_field_id, kind='bar', legend=False, ax=plt.gca())
    plt.title(f"Group-wise Mean of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook has demonstrated how to load, inspect, and process a FAIR^2-compliant dataset using `mlcroissant`, referencing all entities by their `@id` fields for precise data access. The methodologies shown—listing record set and field `@id`s, extracting tabular data, filtering, normalization, grouping, and visualizing—provide a foundation for further in-depth analytic or machine learning workflows. 

**Remember**: For advanced analytics, always tailor your field selections by examining the schema and metadata (`@id`). Consult the original dataset documentation for field definitions and appropriate scientific interpretation.